# Combine dome and horizontal dances with calibrations (both direct transfer and landmark-transfer) locally

video_angle_deg = raw annotated waggle-run angle in the video

true_north_video_deg = where true geographic north is in that same video

waggle_true_bearing_deg = final geographic waggle bearing

1. Load horizontal direct file
2. Load horizontal landmark-transfer file
3. Harmonize column names
4. Combine them into horizontal_all_df
5. Load dome_ALL_calibrated_dances.csv
6. Harmonize dome column names
7. Combine horizontal_all_df + dome_df
8. Run QC

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------------
# Paths
# ---------------------------------------------------------------

HORIZONTAL_DIRECT_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\all_dances_with_calibrations\horizontal_direct_dances_with_calibrations.csv"
)

HORIZONTAL_LANDMARK_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\all_dances_with_calibrations\horizontal_landmark_dances_with_calibration.csv"
)

DOME_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\all_dances_with_calibrations\dome_ALL_calibrated_dances.csv"
)

OUTPUT_HORIZONTAL_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\all_dances_with_calibrations\horizontal_ALL_calibrated_dances.csv"
)

OUTPUT_ALL_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\ALL_horizontal_and_dome_calibrated_waggle_runs.csv"
)

DECLINATION_DEG = -21.27

In [2]:
# ---------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------

def clean_annotation_name(filename):
    if pd.isna(filename):
        return pd.NA

    name = Path(str(filename)).name.strip()

    if name.endswith(".csv"):
        name = name[:-4]

    if name.endswith("_waggle_annotations"):
        name = name.replace("_waggle_annotations", "")

    return name


def add_quality_flag(n):
    n = pd.to_numeric(n, errors="coerce")

    if pd.isna(n):
        return pd.NA
    elif n == 1:
        return "only_one_annotation"
    elif n < 5:
        return "few_annotations"
    else:
        return "ok"


def ensure_column(df, column, default=pd.NA):
    if column not in df.columns:
        df[column] = default
    return df


def fill_column_from(df, target_col, source_col):
    """
    Fill target_col from source_col only where target_col is missing.
    """
    ensure_column(df, target_col, pd.NA)

    if source_col in df.columns:
        df[target_col] = df[target_col].fillna(df[source_col])

    return df


def standardize_shared_column_names(df):
    """
    Makes equivalent columns use the same name.
    """
    rename_map = {
        "calibration_mean_north_deg": "mean_magnetic_north_deg",
        "magnetic_north_calibration_deg": "mean_magnetic_north_deg",
        "calibration_circular_sd_deg": "compass_circular_sd_deg",
        "compass_n_annotations": "calibration_n_annotations",
        "n_compass_annotations": "calibration_n_annotations",
    }

    existing_rename_map = {
        old: new for old, new in rename_map.items()
        if old in df.columns and new not in df.columns
    }

    df = df.rename(columns=existing_rename_map)

    return df


def derive_date_from_source_batch(value):
    """
    Extracts dates like 20250125 from source_batch/dance_key strings.
    """
    if pd.isna(value):
        return pd.NA

    match = re.search(r"(20\d{6})", str(value))

    if match:
        return match.group(1)

    return pd.NA

## Load and standardize the two horizontal files

In [3]:
# ---------------------------------------------------------------
# Load horizontal files
# ---------------------------------------------------------------

horizontal_direct_df = pd.read_csv(
    HORIZONTAL_DIRECT_FILE,
    dtype=str,
    encoding="utf-8-sig"
)

horizontal_landmark_df = pd.read_csv(
    HORIZONTAL_LANDMARK_FILE,
    dtype=str,
    encoding="utf-8-sig"
)

horizontal_direct_df.columns = horizontal_direct_df.columns.str.strip()
horizontal_landmark_df.columns = horizontal_landmark_df.columns.str.strip()

horizontal_direct_df = standardize_shared_column_names(horizontal_direct_df)
horizontal_landmark_df = standardize_shared_column_names(horizontal_landmark_df)

In [4]:
# ---------------------------------------------------------------
# Combine horizontal direct + horizontal landmark-transfer
# ---------------------------------------------------------------

horizontal_columns = list(
    dict.fromkeys(
        list(horizontal_direct_df.columns) + list(horizontal_landmark_df.columns)
    )
)

horizontal_direct_std = horizontal_direct_df.reindex(columns=horizontal_columns)
horizontal_landmark_std = horizontal_landmark_df.reindex(columns=horizontal_columns)

horizontal_all_df = pd.concat(
    [horizontal_direct_std, horizontal_landmark_std],
    ignore_index=True,
    sort=False
)

# Put important columns first
important_horizontal_columns = [
    "recording_type",
    "source_file_type",
    "source_batch",
    "dance_key",
    "dance_id",
    "bee_id",
    "date",
    "time",
    "condition",
    "orientation_method",
    "waggle_run",
    "csv_file",
    "video_name",
    "direction_u",
    "direction_v",
    "video_angle_deg",
    "true_north_video_deg",
    "waggle_true_bearing_deg",
    "mean_magnetic_north_deg",
    "true_north_calibration_deg",
    "compass_circular_sd_deg",
    "calibration_n_annotations",
    "calibration_quality_flag",
    "reference_compass_file",
    "assigned_calibration_file",
    "calibration_id",
    "calibration_file",
    "reference_landmark_file",
    "landmark_dance_file",
    "landmark_calibration_deg",
    "landmark_calibration_circular_sd_deg",
    "north_relative_to_landmark_deg",
    "landmark_dance_video_deg",
    "landmark_dance_circular_sd_deg",
    "start_x",
    "start_y",
    "start_xy",
    "start_frame",
    "annotation_session",
    "notes",
    "orientation_notes",
]

ordered_horizontal_columns = [
    col for col in important_horizontal_columns
    if col in horizontal_all_df.columns
] + [
    col for col in horizontal_all_df.columns
    if col not in important_horizontal_columns
]

horizontal_all_df = horizontal_all_df[ordered_horizontal_columns]

horizontal_all_df.to_csv(
    OUTPUT_HORIZONTAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved combined horizontal file to:\n{OUTPUT_HORIZONTAL_FILE}")

Saved combined horizontal file to:
C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\all_dances_with_calibrations\horizontal_ALL_calibrated_dances.csv


In [5]:
# ---------------------------------------------------------------
# QC for horizontal combined file
# ---------------------------------------------------------------

print("HORIZONTAL QC")
print("Rows:", len(horizontal_all_df))
print("Dances:", horizontal_all_df["dance_key"].nunique())
print("Bees:", horizontal_all_df["bee_id"].nunique())

print("\nRows per orientation_method:")
print(horizontal_all_df["orientation_method"].value_counts(dropna=False))

print("\nDances per orientation_method:")
print(horizontal_all_df.groupby("orientation_method")["dance_key"].nunique())

print("\nMissing key angle columns:")
print(
    horizontal_all_df[
        [
            "video_angle_deg",
            "true_north_video_deg",
            "waggle_true_bearing_deg"
        ]
    ].isna().sum()
)

# Formula check
video_angle = pd.to_numeric(horizontal_all_df["video_angle_deg"], errors="coerce")
true_north = pd.to_numeric(horizontal_all_df["true_north_video_deg"], errors="coerce")
bearing = pd.to_numeric(horizontal_all_df["waggle_true_bearing_deg"], errors="coerce")

expected_bearing = (video_angle - true_north) % 360
difference = ((expected_bearing - bearing + 180) % 360) - 180

print("\nMaximum formula difference:", difference.abs().max())
print("Formula mismatches > 0.000001°:", (difference.abs() > 1e-6).sum())

HORIZONTAL QC
Rows: 1375
Dances: 146
Bees: 27

Rows per orientation_method:
orientation_method
direct_compass       1346
landmark_transfer      29
Name: count, dtype: int64

Dances per orientation_method:
orientation_method
direct_compass       142
landmark_transfer      4
Name: dance_key, dtype: int64

Missing key angle columns:
video_angle_deg            0
true_north_video_deg       0
waggle_true_bearing_deg    0
dtype: int64

Maximum formula difference: 1.1368683772161603e-13
Formula mismatches > 0.000001°: 0


In [6]:
# ---------------------------------------------------------------
# Load and harmonize dome combined file
# ---------------------------------------------------------------

dome_df = pd.read_csv(
    DOME_FILE,
    dtype=str,
    encoding="utf-8-sig"
)

dome_df.columns = dome_df.columns.str.strip()
dome_df = standardize_shared_column_names(dome_df)

dome_df["recording_type"] = "dome"

ensure_column(dome_df, "declination_deg", DECLINATION_DEG)

if "source_batch" not in dome_df.columns:
    if "source_file_type" in dome_df.columns:
        dome_df["source_batch"] = dome_df["source_file_type"]
    else:
        dome_df["source_batch"] = "dome"

fill_column_from(dome_df, "calibration_file", "reference_compass_file")
fill_column_from(dome_df, "assigned_calibration_file", "reference_compass_file")
fill_column_from(dome_df, "calibration_video_name", "reference_compass_file")

ensure_column(dome_df, "calibration_id", pd.NA)

if "reference_compass_file" in dome_df.columns:
    dome_df["calibration_id"] = dome_df["calibration_id"].fillna(
        dome_df["reference_compass_file"].apply(clean_annotation_name)
    )

# For direct-compass rows, true_north_calibration_deg can be filled from true_north_video_deg.
fill_column_from(dome_df, "true_north_calibration_deg", "true_north_video_deg")

if "calibration_quality_flag" not in dome_df.columns:
    dome_df["calibration_quality_flag"] = dome_df[
        "calibration_n_annotations"
    ].apply(add_quality_flag)

In [7]:
# ---------------------------------------------------------------
# Combine horizontal + dome
# ---------------------------------------------------------------
horizontal_all_df["recording_type"] = "horizontal"
dome_df["recording_type"] = "dome"

all_columns = list(
    dict.fromkeys(
        list(horizontal_all_df.columns) + list(dome_df.columns)
    )
)

horizontal_std = horizontal_all_df.reindex(columns=all_columns)
dome_std = dome_df.reindex(columns=all_columns)

all_df = pd.concat(
    [horizontal_std, dome_std],
    ignore_index=True,
    sort=False
)

important_all_columns = [
    "recording_type",
    "source_file_type",
    "source_batch",
    "dance_key",
    "dance_id",
    "bee_id",
    "date",
    "time",
    "condition",
    "aperture",
    "orientation_method",
    "waggle_run",
    "csv_file",
    "video_name",
    "direction_u",
    "direction_v",
    "video_angle_deg",
    "true_north_video_deg",
    "waggle_true_bearing_deg",
    "mean_magnetic_north_deg",
    "true_north_calibration_deg",
    "compass_circular_sd_deg",
    "calibration_n_annotations",
    "calibration_quality_flag",
    "reference_compass_file",
    "assigned_calibration_file",
    "calibration_id",
    "calibration_file",
    "reference_landmark_file",
    "landmark_dance_file",
    "landmark_calibration_deg",
    "landmark_calibration_circular_sd_deg",
    "north_relative_to_landmark_deg",
    "landmark_dance_video_deg",
    "landmark_dance_circular_sd_deg",
    "start_x",
    "start_y",
    "start_xy",
    "start_frame",
    "annotation_session",
    "notes",
    "orientation_notes",
]

ordered_all_columns = [
    col for col in important_all_columns
    if col in all_df.columns
] + [
    col for col in all_df.columns
    if col not in important_all_columns
]

all_df = all_df[ordered_all_columns]

all_df.to_csv(
    OUTPUT_ALL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved final horizontal + dome file to:\n{OUTPUT_ALL_FILE}")

Saved final horizontal + dome file to:
C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\ALL_horizontal_and_dome_calibrated_waggle_runs.csv


In [8]:
# ---------------------------------------------------------------
# Final QC
# ---------------------------------------------------------------

print("FINAL HORIZONTAL + DOME QC")
print("Rows:", len(all_df))
print("Dances:", all_df["dance_key"].nunique())
print("Bees:", all_df["bee_id"].nunique())

print("\nRows per recording_type:")
print(all_df["recording_type"].value_counts(dropna=False))

print("\nRows per orientation_method:")
print(all_df["orientation_method"].value_counts(dropna=False))

print("\nDances per recording_type:")
print(all_df.groupby("recording_type")["dance_key"].nunique())

print("\nDances per orientation_method:")
print(all_df.groupby("orientation_method")["dance_key"].nunique())

print("\nMissing key angle columns:")
print(
    all_df[
        [
            "video_angle_deg",
            "true_north_video_deg",
            "waggle_true_bearing_deg"
        ]
    ].isna().sum()
)

# Formula check
video_angle = pd.to_numeric(all_df["video_angle_deg"], errors="coerce")
true_north = pd.to_numeric(all_df["true_north_video_deg"], errors="coerce")
bearing = pd.to_numeric(all_df["waggle_true_bearing_deg"], errors="coerce")

expected_bearing = (video_angle - true_north) % 360
difference = ((expected_bearing - bearing + 180) % 360) - 180

print("\nMaximum formula difference:", difference.abs().max())
print("Formula mismatches > 0.000001°:", (difference.abs() > 1e-6).sum())

# Duplicate check
duplicate_runs = all_df.duplicated(
    subset=["dance_key", "waggle_run"],
    keep=False
)

print("\nDuplicate dance_key + waggle_run rows:", duplicate_runs.sum())

if duplicate_runs.sum() > 0:
    display(
        all_df.loc[
            duplicate_runs,
            [
                "recording_type",
                "orientation_method",
                "dance_key",
                "waggle_run",
                "csv_file"
            ]
        ].sort_values(["dance_key", "waggle_run"])
    )

FINAL HORIZONTAL + DOME QC
Rows: 2530
Dances: 244
Bees: 33

Rows per recording_type:
recording_type
horizontal    1375
dome          1155
Name: count, dtype: int64

Rows per orientation_method:
orientation_method
direct_compass       2100
landmark_transfer     430
Name: count, dtype: int64

Dances per recording_type:
recording_type
dome           98
horizontal    146
Name: dance_key, dtype: int64

Dances per orientation_method:
orientation_method
direct_compass       203
landmark_transfer     41
Name: dance_key, dtype: int64

Missing key angle columns:
video_angle_deg            0
true_north_video_deg       0
waggle_true_bearing_deg    0
dtype: int64

Maximum formula difference: 1.0231815394945443e-12
Formula mismatches > 0.000001°: 0

Duplicate dance_key + waggle_run rows: 0
